In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# --- Paths & Hyperparams ---
CSV_PATH        = "/home/iambrink/NOH_Thyroid_Cancer_Data/CSV-files/Thyroid_Cancer_TAN&NOH_file.csv"
BASE_IMAGE_PATH = "/home/iambrink/NOH_Thyroid_Cancer_Data/superdata/"

# Swap in your desired MedViT variant here:
# Options: "MedViT_small", "MedViT_base", "MedViT_large" :contentReference[oaicite:0]{index=0}
MODEL_NAME  = "MedViT_small"
NUM_CLASSES = 2
BATCH_SIZE  = 8
NUM_EPOCHS  = 2000
LR          = 5e-3
WD          = 1e-4
NUM_WORKERS = 8
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load & split DataFrame ---
df = pd.read_csv(CSV_PATH).dropna(subset=["Surgery diagnosis in number"])
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df["Surgery diagnosis in number"]
)

# --- Dataset ---
class ThyroidDataset(Dataset):
    def __init__(self, df, base_path, transform=None):
        self.df = df.reset_index(drop=True)
        self.base = base_path
        self.tf   = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img_path = os.path.join(self.base, row["image_path"].replace("\\","/"))
        img = Image.open(img_path).convert("RGB")
        label = int(row["Surgery diagnosis in number"])
        if self.tf:
            img = self.tf(img)
        return img, torch.tensor(label, dtype=torch.long)

# --- Transforms & DataLoaders ---
# Resolve the right resize/normalization for MedViT (ImageNet defaults)
config          = resolve_data_config({}, model=None)
train_transform = create_transform(**config, is_training=True)
val_transform   = create_transform(**config, is_training=False)

train_ds = ThyroidDataset(train_df, BASE_IMAGE_PATH, train_transform)
val_ds   = ThyroidDataset(val_df,   BASE_IMAGE_PATH, val_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader   = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# --- Model & freeze backbone ---
# Instantiate MedViT via timm’s registry
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
).to(DEVICE)

# Freeze all except the final classifier head
for p in model.parameters():
    p.requires_grad = False
for p in model.get_classifier().parameters():
    p.requires_grad = True

# --- Loss, optimizer, scheduler, scaler ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WD
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.cuda.amp.GradScaler()
torch.backends.cudnn.benchmark = True

best_val_acc = 0.0

# --- Training & Validation ---
for epoch in range(1, NUM_EPOCHS + 1):
    # — Train —
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            patch_logits = model(imgs)      # [B, num_patches, 2] for MedViT
            logits       = patch_logits.mean(dim=1)   # [B,2]
            loss         = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)

    scheduler.step()
    avg_train_loss = total_loss / len(train_ds)

    # — Validate —
    model.eval()
    val_loss = 0.0
    correct  = 0
    total    = 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast():
                patch_logits = model(imgs)
                logits       = patch_logits.mean(dim=1)
                loss         = criterion(logits, labels)

            val_loss += loss.item() * imgs.size(0)
            preds    = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    avg_val_loss = val_loss / len(val_ds)
    val_acc      = correct / total

    print(
        f"Epoch {epoch:2d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss:   {avg_val_loss:.4f} | "
        f"Val Acc:    {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/home/iambrink/NOH_Thyroid_Cancer_Data/MODELS/best_medvit4.pth")
        print(f"→ Saved new best model (Acc: {best_val_acc:.4f})")


In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import torch
from torch.utils.data import Dataset, DataLoader
import timm

# --- Paths & Hyperparams (reuse from training) ---
CSV_PATH        = "/home/iambrink/NOH_Thyroid_Cancer_Data/CSV-files/Thyroid_Cancer_TAN&NOH_file.csv"
BASE_IMAGE_PATH = "/home/iambrink/NOH_Thyroid_Cancer_Data/superdata/"
MODEL_NAME      = "MedViT_small"    # or "MedViT_base" / "MedViT_large"
NUM_CLASSES     = 2
BATCH_SIZE      = 8
NUM_WORKERS     = 8
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Dataset class (same as before) ---
class ThyroidDataset(Dataset):
    def __init__(self, df, base_path, transform=None):
        self.df = df.reset_index(drop=True)
        self.base = base_path
        self.tf   = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img_path = os.path.join(self.base, row["image_path"].replace("\\","/"))
        img = Image.open(img_path).convert("RGB")
        label = int(row["Surgery diagnosis in number"])
        if self.tf:
            img = self.tf(img)
        return img, torch.tensor(label, dtype=torch.long)

# --- Prepare test split & transforms ---
df = pd.read_csv(CSV_PATH).dropna(subset=["Surgery diagnosis in number"])
_, test_df = train_test_split(
    df, test_size=0.1, random_state=42,
    stratify=df["Surgery diagnosis in number"]
)

from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
config         = resolve_data_config({}, model=None)
test_transform = create_transform(**config, is_training=False)

test_ds     = ThyroidDataset(test_df, BASE_IMAGE_PATH, test_transform)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# --- Load checkpoint ---
model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES
).to(DEVICE)
model.load_state_dict(torch.load("best_medvit.pth"))
model.eval()

# --- Run inference & collect preds, labels, probs ---
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Testing"):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        patch_logits = model(imgs)               # [B, num_patches, C]
        logits       = patch_logits.mean(dim=1)  # [B, C]
        probs        = torch.softmax(logits, dim=1)

        preds = logits.argmax(dim=1)
        # record
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # positive‑class prob

# --- Compute metrics ---
acc   = accuracy_score(all_labels, all_preds)
prec  = precision_score(all_labels, all_preds)
rec   = recall_score(all_labels, all_preds)
f1    = f1_score(all_labels, all_preds)
auc   = roc_auc_score(all_labels, all_probs)

print(f"Test Accuracy : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1‑Score      : {f1:.4f}")
print(f"AUC           : {auc:.4f}")

# (Optional) Confusion matrix & classification report
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))
